<a href="https://colab.research.google.com/github/muhammeddanisht/langgraph-agents/blob/main/langgraph_v3_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# Check versions — no install needed, already have from v2
!pip install langgraph langchain-groq langchain-core --quiet
import langgraph
import langchain_groq
import langchain_core

import importlib.metadata

print(importlib.metadata.version("langgraph"))
print(importlib.metadata.version("langchain-groq"))
print(importlib.metadata.version("langchain-core"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.6 MB/s eta 0:00:00
1.1.9
1.1.2
1.3.1


In [7]:
# ── IMPORTS ──────────────────────────────────────────────
from langchain_core.tools import tool          # @tool stamp
from langchain_groq import ChatGroq            # Groq brain (same as v2)
from langchain_core.messages import HumanMessage  # label for our message
from langgraph.graph import StateGraph, MessagesState, START, END  # graph pieces
from langgraph.prebuilt import ToolNode, tools_condition  # worker room + signal
from langgraph.checkpoint.memory import MemorySaver  # memory (same as v2)
import os
from google.colab import userdata

# ── GROQ SETUP ───────────────────────────────────────────
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# ── TOOLS (2 workers) ────────────────────────────────────
@tool
def calculator(a: float, b: float, operation: str) -> float:
    """Do math. operation must be: add, subtract, multiply, divide."""
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        return a / b

@tool
def get_weather(city: str) -> str:
    """Get the weather for a city. Use when user asks about weather."""
    # Fake data
    weather = {"kerala": "32°C sunny", "delhi": "41°C hot", "mumbai": "28°C humid"}
    return weather.get(city.lower(), "Weather not found for that city")

# List of all workers
tools = [calculator, get_weather]

In [8]:
# ── LLM WITH TOOLS ───────────────────────────────────────
llm = ChatGroq(model="llama-3.1-8b-instant")
llm_with_tools = llm.bind_tools(tools)  # manager gets worker list

# ── NODES ────────────────────────────────────────────────
def agent_node(state: MessagesState):
    # manager thinks using llm_with_tools (not plain llm)
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": response}

# worker room — all tools sit here
tool_node = ToolNode(tools, handle_tool_errors=True)

# ── BUILD GRAPH ──────────────────────────────────────────
graph = StateGraph(MessagesState)
graph.add_node("agent", agent_node)   # manager room
graph.add_node("tools", tool_node)    # worker room

graph.add_edge(START, "agent")        # start → manager

graph.add_conditional_edges(          # traffic signal
    "agent",                          # FROM: manager room
    tools_condition,                  # SIGNAL: tool call? yes/no
)

graph.add_edge("tools", "agent")      # worker done → back to manager

memory = MemorySaver()
app = graph.compile(checkpointer=memory)

In [9]:
config = {"configurable": {"thread_id": "test1"}}

result = app.invoke(
    {"messages": [HumanMessage(content="What is 25 multiplied by 4?")]},
    config=config
)

print(result["messages"][-1].content)

The result of the multiplication is 100.


In [10]:
result = app.invoke(
    {"messages": [HumanMessage(content="What is the weather in Kerala?")]},
    config=config
)

print(result["messages"][-1].content)

The weather in Kerala is currently 32°C with sunny conditions.
